# Webページから情報を取り出してみよう

きょうは **スクレイピング** に挑戦します。
スクレイピング＝Webページを読みこんで、ほしい部分だけを取り出すこと。

材料はこの2つだけ。むずかしい準備はいりません。

- `requests` … ページを取ってくる係
- `BeautifulSoup` … 取ってきたページをバラバラに分解して、探しやすくする係


### 講師メモ：全体の流れ

| ステップ | 内容 | 目安 |
|---|---|---|
| 0 | ブラウザでサイトを見る | 5分 |
| 1 | ページのタイトルを取る | 10分 |
| 2 | 商品名を全部並べる | 15分 |
| 3 | 価格を取って最安値を探す | 15分 |
| 4 | 詳細ページを辿る（発展） | 進度差の吸収用 |

- ステップ3まで終われば授業として成立する。ステップ4は早く終わった生徒用。
- 各コードセルは上から順に実行する前提。飛ばすと `NameError` になる。


## 準備

最初に1回だけ実行するセルです。
左の ▶ ボタンを押すか、`Shift + Enter` を押すと実行できます。


In [ ]:
import requests
from bs4 import BeautifulSoup

# 練習用サイトのアドレス。これ以降このURLを直接書くことはありません
BASE_URL = "https://yuracode.github.io/shop/"

print("準備できました")

### 講師メモ

- Colab には `requests` も `bs4` も最初から入っている。`pip install` は不要。
- `import` を忘れると以降すべて `NameError`。実行し忘れが最も多い事故なので、
  ここで全員の画面に「準備できました」が出たことを目視で確認する。
- URL を書くのはこのセルだけ。以降は `BASE_URL` を使い回す。


## ステップ0：まず相手を見る

コードを書く前に、これから読みこむページをブラウザで開いてみましょう。

https://yuracode.github.io/shop/

「サイバー購買部」というお店のページです。商品が8つ並んでいますね。
このページから、商品名や値段をプログラムで取り出すのが今日のゴールです。


### 講師メモ

- ここではコードを書かせない。**相手を見てから触る**という順番を体で覚えてもらう。
- 余裕があれば右クリック →「検証」で HTML を見せる。
  `<div class="item">` が8個ならんでいることだけ指させれば十分。
  タグの文法説明には入らないこと。


## ステップ1：ページのタイトルを取る

まずは小さく成功しましょう。ページの「タイトル」だけを取り出します。

やることは2つ。**ページを取ってくる** → **タイトルを探す**。


### 1-1. ページを取ってくる

`requests.get(...)` で、指定したアドレスのページを取ってきます。


In [ ]:
res = requests.get(BASE_URL)
res.encoding = "utf-8"   # 日本語が文字化けしないようにするおまじない
html = res.text

### 1-2. 取れたものを見る

`html` の中身を、最初の300文字だけのぞいてみましょう。


In [ ]:
print(html[:300])

記号だらけで読みにくいですね。これが HTML です。

このままでは探しにくいので、`BeautifulSoup` に渡して**分解**してもらいます。
分解しておくと「タイトルを持ってきて」といった頼み方ができるようになります。


In [ ]:
soup = BeautifulSoup(html, "html.parser")

### 1-3. タイトルを取り出す

`soup.title` でタイトルの部分、`.text` でその中の文字だけが取れます。


In [ ]:
print(soup.title.text)

### 講師メモ

- `res.encoding = "utf-8"` は文字化け防止。生徒には「おまじない」で流してよい。
  聞かれたら「日本語をどう読むかをこちらから指定している」とだけ答える。
- `soup.title` だけだと `<title>サイバー購買部</title>` とタグごと出る。
  `.text` を付けると中身だけになる、という差をその場で見せると理解が早い。
- 「取れた！」を必ず声に出して確認する。ここが今日最初の成功体験。


### やってみよう

`title` を `h1` に変えると、ページの大きな見出しが取れます。


In [ ]:
print(soup.h1.text)

## ステップ2：商品名を全部ならべる

つぎは商品名です。8つあるので、**まとめて全部**取ります。

このサイトでは、商品1つ分が `<div class="item">` という箱に入っています。
`find_all` を使うと、その箱を**全部**まとめて取ってこられます。


In [ ]:
items = soup.find_all("div", class_="item")

いくつ取れたか数えてみましょう。`len(...)` は個数を数える命令です。


In [ ]:
print(len(items))

### 講師メモ

- `8` が出れば成功。ここで数が合わないときは `class_` のつづり間違いがほぼ100%。
- `find` は最初の1つだけ、`find_all` は全部。ここは板書して区別させる。
- `class_` のアンダースコアを落とすとエラーになる。Python の予約語 `class` を
  避けるための決まりごと、と一言だけ添える（詳しい説明には踏みこまない）。


### 2-2. 箱の中から名前を取り出す

`items` の中身を1つずつ順番に見ていきます。これが**くり返し（ループ）**です。

商品名は `<h2 class="item-name">` に入っています。


In [ ]:
for item in items:
    name = item.find("h2", class_="item-name")
    print(name.text)

### 講師メモ

- 8行ずらっと出た瞬間がいちばん盛り上がるところ。手を止めて画面を見せ合わせる。
- `for` の下の行が字下げ（インデント）されている点に触れる。
  全角スペースが混ざると `IndentationError`。発生したら全員に注意喚起する。


### やってみよう

`items[0]` は「1つめの商品」という意味です。
`0` を `1` や `2` に変えて、別の商品の名前を出してみましょう。


In [ ]:
print(items[0].find("h2", class_="item-name").text)

## ステップ3：値段も取って、最安値を探す

名前が取れたので、つぎは値段です。
値段は `<span class="item-price">` に、数字だけが入っています。


In [ ]:
for item in items:
    name = item.find("h2", class_="item-name").text
    price = item.find("span", class_="item-price").text
    print(name, price)

### 3-2. 文字を数字に変える

いま取れた `150` は、じつは**文字**であって数ではありません。
文字のままだと大小をくらべられないので、`int(...)` で数に変えます。


In [ ]:
price_text = items[0].find("span", class_="item-price").text
price = int(price_text)

print(price_text, "→", price)

### 講師メモ

- 画面上はどちらも `150` に見えるので、差が伝わりにくい。
  `print(price_text + 10)` を実演してエラーを見せると腹落ちする（文字と数はたし算できない）。
- HTML 側であえて「150円」ではなく「150」だけを入れてある。
  文字列処理の話に脱線させないための設計、と押さえておく。


### 3-3. 一番安い商品を探す

順番に見ていって、「いままでで一番安い値段」より安ければ覚えなおす。
これをくり返せば、最後に残るのが最安値です。


In [ ]:
cheapest_name = ""
cheapest_price = 99999

for item in items:
    name = item.find("h2", class_="item-name").text
    price = int(item.find("span", class_="item-price").text)
    if price < cheapest_price:
        cheapest_price = price
        cheapest_name = name

In [ ]:
print("一番安いのは", cheapest_name, "で", cheapest_price, "円です")

### 講師メモ

- `cheapest_price = 99999` の意味（最初はありえない大きな数を置く）を口頭で補う。
- ここまでで45分程度。**授業としてはここで完結してよい。**
- 早く終わった生徒にはステップ4へ進ませ、そうでない生徒はやってみようを触らせる。


### やってみよう

`<` を `>` に、`99999` を `0` に変えると、こんどは**一番高い商品**が探せます。


## ステップ4：詳細ページを辿る（発展）

ここからは発展課題です。ここまでで終わっても大丈夫。

一覧ページには**在庫**が載っていません。在庫は各商品の詳細ページにあります。
そこで「リンクをたどって、その先のページも読む」ということをします。


### 4-1. リンク先のアドレスを取り出す

`<a href="...">` の `href` の部分が、リンク先のアドレスです。


In [ ]:
for item in items:
    link = item.find("a", class_="item-link")
    print(link.get("href"))

`items/item01.html` のように、途中からのアドレスが出てきました。
`BASE_URL` とくっつけると、完全なアドレスになります。


### 4-2. 1つだけ詳細ページを開いてみる


In [ ]:
href = items[0].find("a", class_="item-link").get("href")

detail_res = requests.get(BASE_URL + href)
detail_res.encoding = "utf-8"
detail_soup = BeautifulSoup(detail_res.text, "html.parser")

In [ ]:
print(detail_soup.find("p", class_="item-stock").text)

### 講師メモ

- ここでやっていることはステップ1と同じ（取ってくる → 分解する → 探す）。
  「新しいことは何もしていない」と言い切ってしまってよい。
- `BASE_URL + href` の連結は、`BASE_URL` が `/` で終わっているから成立している。


### 4-3. 8商品ぶん、順番に見にいく

最後に、全部の商品の在庫を調べます。

`time.sleep(1)` という行があります。これは**1秒待つ**という命令です。
相手のサーバーに一気に何回もお願いすると迷惑になるので、間を空けています。


In [ ]:
import time

for item in items:
    name = item.find("h2", class_="item-name").text
    href = item.find("a", class_="item-link").get("href")

    detail_res = requests.get(BASE_URL + href)
    detail_res.encoding = "utf-8"
    detail_soup = BeautifulSoup(detail_res.text, "html.parser")

    stock = detail_soup.find("p", class_="item-stock").text

    print(name, ":", stock)
    time.sleep(1)

### 講師メモ

- 8秒ほどかかる。「待たされている」という体感そのものが、
  次のマナーの話（負荷をかけるとはどういうことか）への導入になる。
- `time.sleep(1)` を消したらどうなるか、を**口頭で**問いかけるだけにとどめる。
  実際に外部サイトへ試させないこと。


## おわりに

きょうやったことは、たった3つのくり返しでした。

1. `requests.get(...)` でページを取ってくる
2. `BeautifulSoup(...)` で分解する
3. `find` / `find_all` でほしい場所を探す

相手のサイトが変わっても、やることはこれだけです。おつかれさまでした。
